
# Phase 13 — Lifecycle of Persistent Weakness

**Historical characterization and candidate generation. Not a new external validation and not a trading strategy.**

Phase 12 tested three predeclared explanations for the small Day-20 rebound relationship:

- peer confirmation;
- rebound concentration;
- aggregate single-stock futures OI change.

None was supported strongly enough to carry forward.

Phase 13 therefore changes the outcome rather than adding more indicators.

> **Once a stock has remained weak versus both Nifty 50 and its sector for 20 consecutive sessions, what prospectively observable Day-20 state variables are associated with how quickly the weakness episode ends?**

## Primary exploratory candidate

`sessions_since_trailing20_stock_trough`

Phase 12A suggested, after the result was already viewed, that a **recent trough** may be associated with a weakness episode remaining active longer.

The frozen Phase 13 directional candidate is therefore:

> **Older troughs should be associated with a higher probability that the original weakness episode ends within the next 10 market sessions.**

Because this clue was discovered post hoc in Phase 12A, historical support here is **candidate-generation evidence only**. A surviving candidate must later be tested prospectively.

## Primary outcome

`episode_ends_within_10d_after_day20`

The original episode definition is retained. No new "recovery" threshold is invented.

## Primary model

Clustered logistic regression:

`10D episode exit ~ trough age + rebound + current severity + prior maximum severity + year`

The primary expected sign on **trough age** is positive.

The specification, endpoint, controls, bootstrap plan, and promotion rule were written to GitHub before this notebook was run.


In [ ]:

# ============================================================
# 1. SETUP + FROZEN UNIVERSES + EXECUTION LOCK
# ============================================================

!pip -q install --upgrade yfinance nse-archives scipy statsmodels patsy lifelines

from io import BytesIO
from pathlib import Path
import json
import time
import zipfile
import warnings

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from scipy.stats import spearmanr
import statsmodels.formula.api as smf
import patsy
from nsedata import nse

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# ---------------- FROZEN FROM PHASE 10 ----------------
START_DATE = "2018-01-01"
END_DATE = "2026-08-22"   # yfinance end is exclusive; last allowed date = 2026-08-21

LANDMARK_DAY = 20
TRAILING_REBOUND_WINDOW = 20
PRIMARY_FORWARD_HORIZON = 40
MAX_FORWARD_HORIZON = 60

PRIMARY_PREDICTOR = "stock_rebound_trailing20_from_low"
PRIMARY_OUTCOME = "future_rel_sector_40d"

NIFTY_SYMBOL = "^NSEI"
BATCH_SIZE = 40
MAX_RETRIES = 3
MIN_VALID_SECTOR_PEERS = 2

# Phase 10 published reference values. These are used only as a reconstruction gate.
PHASE10_REFERENCE_N = 1386
PHASE10_REFERENCE_UNIQUE_STOCKS = 352
PHASE10_REFERENCE_RHO = -0.0611
BASELINE_N_TOLERANCE = 30
BASELINE_STOCK_TOLERANCE = 12
BASELINE_RHO_TOLERANCE = 0.015

# ---------------- PHASE 13 LOCK ----------------
BOOTSTRAPS = 2000
BOOTSTRAP_SEED = 20260825
PRIMARY_EXIT_HORIZON = 10
SECONDARY_EXIT_HORIZONS = [5, 20]

REPO_RAW = "https://raw.githubusercontent.com/chinmay227/indian-market-internals/main"
DISCOVERY_URL = f"{REPO_RAW}/data/reference/sampled_nifty500_universe_100.csv"
SECTOR_SNAPSHOT_URL = f"{REPO_RAW}/data/reference/nifty500_sector_universe_snapshot.csv"
PHASE13_SPEC_URL = f"{REPO_RAW}/reports/phase_13_lifecycle_of_persistent_weakness_spec.md"

def read_csv_url(url):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    r.raise_for_status()
    return pd.read_csv(BytesIO(r.content))

def read_text_url(url):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    r.raise_for_status()
    return r.text

discovery = read_csv_url(DISCOVERY_URL)
full_universe = read_csv_url(SECTOR_SNAPSHOT_URL)
phase13_spec_text = read_text_url(PHASE13_SPEC_URL)

assert len(discovery) == 100
assert discovery["ticker"].nunique() == 100
assert len(full_universe) == 500
assert full_universe["ticker"].nunique() == 500

validation_universe = (
    full_universe[~full_universe["ticker"].isin(discovery["ticker"])]
    .copy()
    .reset_index(drop=True)
)

assert len(validation_universe) == 400
assert validation_universe["ticker"].nunique() == 400
assert set(validation_universe["ticker"]).isdisjoint(set(discovery["ticker"]))

full_universe["yf_ticker"] = full_universe["ticker"] + ".NS"
ticker_to_yf = dict(zip(full_universe["ticker"], full_universe["yf_ticker"]))
yf_to_ticker = {v: k for k, v in ticker_to_yf.items()}

print("=" * 100)
print("PHASE 13 — LIFECYCLE OF PERSISTENT WEAKNESS")
print("=" * 100)
print("Historical characterization only")
print("Validation-universe stocks:", len(validation_universe))
print("Anchor:", f"Day {LANDMARK_DAY}")
print("Primary exit horizon:", PRIMARY_EXIT_HORIZON)
print("Primary candidate: sessions_since_trailing20_stock_trough")
print("Expected trough-age sign: positive")
print("Bootstraps:", BOOTSTRAPS)
